# Single Cell Simulation Example with OBI-One Form Logic with parameter scan over neuronal modification

This notebook demonstrates how to run a single cell simulation using the OBI-One form-based workflow, similar to the circuit simulation example. The single cell scan configuration has a parameter scan over neuronal modifications.

In [ ]:
from entitysdk import Client
from obi_auth import get_token

from obi_notebook import get_projects
from obi_notebook import get_entities
import obi_one as obi

token = get_token(environment="production", auth_mode="daf")
project_context = get_projects.get_projects(token)

db_client = Client(environment="production", project_context=project_context, token_manager=token)

In [3]:
# Optional: Download using unique ID
entity_ID = "6e8cc802-5ce4-4101-9f2d-d8b8cf763e3c"  # <<< FILL IN UNIQUE MEModel ID HERE

if entity_ID != "<MEMODEL-ID>":
    memodel_ids = [entity_ID]
else:
# Alternative: Select from a table of entities
    memodel_ids = []
    memodel_ids = get_entities.get_entities("memodel", token, memodel_ids,
                                            project_context=project_context,
                                            multi_select=False,
                                            default_scale="small")



Here we load some more classes that we need to define our simulation scan configuration

In [4]:
from obi_one.core.info import Info
from obi_one.scientific.blocks.neuronal_manipulations.neuronal_manipulations import (
    ByNeuronMechanismVariableNeuronalManipulation,
    BySectionListMechanismVariableNeuronalManipulation,
    ByNeuronModification,
    BySectionListModification
)
from obi_one.scientific.blocks.recording import SomaVoltageRecording
from obi_one.scientific.blocks.stimuli.stimulus import ConstantCurrentClampSomaticStimulus
from obi_one.scientific.from_id.memodel_from_id import MEModelFromID
from obi_one.scientific.tasks.generate_simulations.config.me_model import MEModelSimulationScanConfig

Now we need to define our simulation scan configuration. For the neuronal modification, we need to give it the id of an ion channel that is already present in the me-model, with the appropriate channel and variable names. Please modify those if you changed the me-model id.

In [5]:
ic_ca_hva2_id = "21ebb7ab-b41b-441d-b494-6665075d26b0"
channel_name="Ca_HVA2"
variable_name="gCa_HVAbar_Ca_HVA2"

sim_conf = MEModelSimulationScanConfig(
    initialize=MEModelSimulationScanConfig.Initialize(
        circuit=MEModelFromID(id_str=memodel_ids[0]),
        simulation_length=200,
        v_init=-80,
        random_seed=1,
    ),
    info=Info(
        campaign_name="MEModel with neuron modifications Campaign Test 001",
        campaign_description="neuron modifications",
    ),
    neuronal_manipulations={
        "manip1": ByNeuronMechanismVariableNeuronalManipulation(
            neuron_set=None,
            modification=ByNeuronModification(
                ion_channel_id=ic_ca_hva2_id,
                channel_name=channel_name,
                variable_name=variable_name,
                variable_type="RANGE",
                # new_value=0.1,  # should enable list here
                new_value=[0.1, 0.2],  # should enable list here
            )
        ),
        "manip2": BySectionListMechanismVariableNeuronalManipulation(
            neuron_set=None,
            modification=BySectionListModification(
                ion_channel_id=ic_ca_hva2_id,
                variable_name=variable_name,
                section_list_modifications={
                    "axonal": [0.3, 0.4],
                }
            )
        )
    },
    stimuli={
        "step1": ConstantCurrentClampSomaticStimulus(
            duration=100,
            neuron_set=None,
            timestamp_offset=20,
            amplitude=0.1,
        )
    },
    recordings={
        "rec_voltage": SomaVoltageRecording(
            neuron_set=None,
            dt=0.1,
        )
    },
    timestamps={}
)

# Validated Config
validated_sim_conf = sim_conf.validated_config()

In [ ]:
grid_scan = obi.GridScanGenerationTask(form=validated_sim_conf, coordinate_directory_option="ZERO_INDEX", output_root='../../../obi-output/memodel_simulations/grid_scan')
grid_scan.multiple_value_parameters(display=True)
grid_scan.coordinate_parameters(display=True)
grid_scan.execute(db_client=db_client)
obi.run_tasks_for_generated_scan(grid_scan, db_client=db_client)